# Лаборатория 09 — Корректировки: защита открытых позиций

Ради этого инструмент и существует. Мы берём две позиции из прошлых модулей — **короткий пут** и
**айрон кондор** на DEMO, — проигрываем неблагоприятные движения через `analyzer.scenario_grid` +
`viz.plot_pnl_heatmap`, а затем **реализуем** каждую корректировку как новую `Position`: закрываем
старые ноги по модельным ценам через `pricing.bsm_price`, открываем новые и сравниваем **выплаты,
греки и POP** до и после.

Три корректировки: (A) ролл протестированного короткого пута **вниз и по времени за кредит**;
(B) ролл **нетронутой колл-стороны** кондора **вниз за кредит**; (C) **конверсия пробитой**
пут-стороны кондора **в бабочку**, чтобы ограничить убыток. Работает офлайн, сверху вниз.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, greeks, viz, pricing
from optionslab.position import Position, OptionLeg, StockLeg
r = lambda x: round(float(x), 2)

def bank_credit(pos, credit):
    """Вложить реализованный кредит ($) от закрытых ног в себестоимость для анализа кампании."""
    legs = list(pos.legs)
    for i, l in enumerate(legs):
        if isinstance(l, OptionLeg) and l.quantity < 0:
            bumped = l.premium + credit / (abs(l.quantity) * l.multiplier)
            legs[i] = OptionLeg(l.kind, l.strike, l.expiry, l.quantity, bumped)
            break
    return Position(tuple(legs), pos.label + " (себестоимость кампании)")

## Сценарий A — протестированный короткий пут

Вы продали 95-й пут на DEMO на 45 DTE за 1.58 (обеспеченный деньгами пут; вы были умеренно бычьими и
не против получить бумагу по 95). С тех пор DEMO упал. Сначала проиграем ущерб: сетка сценариев P&L
по более низким ценам и прошедшим дням.

In [ ]:
sp = strategies.cash_secured_put((95, 1.58), expiry=45/365)
grid = analyzer.scenario_grid(sp, spots=[88, 91, 94, 97, 100], days_forward=[0, 10, 20], vol=0.30)
ax = viz.plot_pnl_heatmap(grid)
ax.set_title("Короткий 95-й пут: P&L при падении DEMO (неблагоприятное движение)")

Красный левый нижний угол и есть проблема: падение к 88 вместе с прошедшим временем — это
растущий убыток. Зададим **текущее** неблагоприятное состояние — спот 92, прошло 20 дней (осталось
25 DTE), IV подскочила до 0.32 — и проверим, **цел ли тезис**. Если вы всё ещё рады получить DEMO
ниже — корректируйте; если причина, по которой вы продали пут, исчезла — закрывайтесь. Здесь мы
считаем, что тезис держится.

In [ ]:
spot, rem, vol = 92.0, 25/365, 0.32
buyback = pricing.bsm_price("put", spot, 95, rem, vol)   # закрываем старую ногу по модельной цене
realized = (buyback - 1.58) * (-1) * 100                 # короткая нога: (закрытие-вход)*кол-во*мульт
orig_now = strategies.cash_secured_put((95, 1.58), expiry=rem)   # сделка как есть, 25 DTE
print("выкуп 95p:", r(buyback), "| реализованный P&L при закрытии $:", r(realized))
print("исходный POP@92:", r(analyzer.probability_of_profit(orig_now, spot, vol)),
      "| дельта $:", r(greeks.position_greeks(orig_now, spot, vol).delta))

Короткий пут в минусе примерно на \$327 и несёт **+63 дельты** (по мере ухода в ITM он всё
больше ведёт себя как длинная акция). **Ролл вниз и по времени за кредит:** выкупаем 95-й пут,
продаём **90-й** пут в более дальнем цикле (120 DTE). Достаточный сдвиг по времени финансирует увод
страйка вниз и всё равно приносит деньги — правило «ролл за кредит».

In [ ]:
new_prem = pricing.bsm_price("put", spot, 90, 120/365, vol)   # открываем новую ногу по модельной цене
roll_net = (new_prem - buyback) * 100
adj = strategies.cash_secured_put((90, round(new_prem, 2)), expiry=120/365)
print("продаём новый 90p:", r(new_prem), "| чистые деньги ролла $ (кредит, если > 0):", r(roll_net))
print("скорр. POP@92:", r(analyzer.probability_of_profit(adj, spot, vol)),
      "| дельта $:", r(greeks.position_greeks(adj, spot, vol).delta),
      "| max_loss $:", r(analyzer.max_loss(adj)))

Ролл даёт **чистый кредит**, понижает страйк обязательства 95 → 90, режет дельту 63 → 41 и
поднимает POP вперёд 0.41 → 0.66. Уже потерянные ~\$327 — **утопленные издержки**; решать надо,
хороша ли *новая* сделка, — а она хороша. Сравним две кривые выплат (новая сдвинута на реализованный
убыток, который вы в неё переносите).

In [ ]:
spots = np.linspace(80, 105, 121)
before = payoff.pnl_curve(orig_now, spots)
after = payoff.pnl_curve(adj, spots) + realized      # кампания с учётом утопленного убытка ролла
fig, ax = plt.subplots()
ax.plot(spots, before, label="ничего не делать (короткий 95p, 25 DTE)")
ax.plot(spots, after, label="ролл в 90p на 120 DTE (+ утопленный убыток)")
ax.axhline(0, color="k", lw=.7); ax.axvline(92, color="grey", lw=.6)
ax.legend(); ax.set_title("Сценарий A: до и после ролла")

## Сценарий B — кондор, пут-сторона протестирована: ролл *нетронутой* колл-стороны вниз

Флагманский кондор из модуля 05: айрон кондор на DEMO 87.5 / 92.5 / 107.5 / 112.5 на 45 DTE, кредит
\$140. DEMO сполз к 93 (осталось 25 DTE, IV 0.30) — **пут**-сторона протестирована. Сначала
проиграем это, затем защитимся, роллируя **безопасную колл-сторону вниз** за кредит.

In [ ]:
ic = strategies.iron_condor((87.5,0.37),(92.5,1.01),(107.5,1.19),(112.5,0.43), expiry=25/365)
grid = analyzer.scenario_grid(ic, spots=[89,92,95,100,105], days_forward=[0,10,20], vol=0.30)
ax = viz.plot_pnl_heatmap(grid); ax.set_title("Айрон кондор: пут-сторона протестирована, DEMO падает к 93")

In [ ]:
spot, rem, vol = 93.0, 25/365, 0.30
# закрываем старый колл-спред по модельным ценам; открываем более низкий 102.5/107.5
bb = pricing.bsm_price("call", spot, 107.5, rem, vol); sl = pricing.bsm_price("call", spot, 112.5, rem, vol)
old_call_realized = (1.19-0.43)*100 - (bb-sl)*100        # собранный кредит по коллам минус стоимость закрытия
new_sc = pricing.bsm_price("call", spot, 102.5, rem, vol); new_lc = pricing.bsm_price("call", spot, 107.5, rem, vol)
roll_net = (new_sc-new_lc)*100 - (bb-sl)*100
print("реализовано при закрытии старого колл-спреда $:", r(old_call_realized), "| чистые деньги ролла $:", r(roll_net))

In [ ]:
adj = strategies.iron_condor((87.5,0.37),(92.5,1.01),
        (102.5, round(new_sc,2)), (107.5, round(new_lc,2)), expiry=rem)
adjc = bank_credit(adj, old_call_realized)               # вкладываем собранные деньги в себестоимость
for name, p in [("исходный", ic), ("с роллом", adjc)]:
    s = analyzer.summarize(p, spot, vol)
    print(f"{name:9} credit={-s['net_premium']:6.0f} BEs={[r(x) for x in s['breakevens']]} "
          f"POP={s['probability_of_profit']:.2f} maxL={r(s['max_loss'])}")

Читаем цифры: ролл кладёт в карман ещё ~\$22 (кредит кампании \$140 → \$162), что
**понижает точку безубыточности на протестированной стороне 91.10 → 90.88** (больше пространства
вниз), **режет максимальный убыток \$360 → \$338** и выравнивает дельту. Платите вы пространством
наверху — верхняя точка безубыточности падает 108.9 → 104.1, а POP слегка проседает. В этом и
размен: снять прибыль безопасной стороны, чтобы защитить протестированную, приняв более узкий верх.
Подтягивайте нетронутую сторону внутрь **только** до тех пор, пока она платит и не становится новой
проблемой.

## Сценарий C — пут-сторона *пробита*: конверсия в бабочку, чтобы ограничить убыток

DEMO продолжает падать до 92 (20 DTE, IV 0.31): короткий 92.5-й пут пробит, и ролла за кредит больше
нет. **Превращаем пробитый короткий пут-спред в длинную пут-бабочку**, продав ещё один 92.5-й пут и
купив 97.5-й пут, — разбежавшаяся короткая нога становится телом бабочки. Это стоит дебета
(разрешённое исключение из правила кредита): вы *покупаете крышку на убыток*.

In [ ]:
spot, rem, vol = 92.0, 20/365, 0.31
put_spread = strategies.custom(OptionLeg("put",87.5,rem,1,0.37), OptionLeg("put",92.5,rem,-1,1.01),
                               label="пробитый пут-спред")
add_sell = pricing.bsm_price("put", spot, 92.5, rem, vol)   # продаём ещё один пут в тело
add_buy = pricing.bsm_price("put", spot, 97.5, rem, vol)    # покупаем верхнее крыло
print("добавляем ноги: продаём 92.5p", r(add_sell), "покупаем 97.5p", r(add_buy),
      "| чистый дебет $:", r((add_buy-add_sell)*100))

In [ ]:
fly = strategies.custom(
    OptionLeg("put", 87.5, rem, 1, 0.37),
    OptionLeg("put", 92.5, rem, -2, round((1.01+add_sell)/2, 2)),   # оба коротких 92.5-х пута
    OptionLeg("put", 97.5, rem, 1, round(add_buy, 2)),
    label="пут-бабочка после конверсии 87.5/92.5/97.5")
for name, p in [("пробитый спред", put_spread), ("бабочка (конв.)", fly)]:
    print(f"{name:16} maxL={r(analyzer.max_loss(p)):>8} maxP={r(analyzer.max_profit(p)):>7} "
          f"POP@92={analyzer.probability_of_profit(p, spot, vol):.2f}")

In [ ]:
spots = np.linspace(82, 100, 121)
fig, ax = plt.subplots()
ax.plot(spots, payoff.pnl_curve(put_spread, spots), label="пробитый спред (без крышки)")
ax.plot(spots, payoff.pnl_curve(fly, spots), label="конверсия в пут-бабочку (с крышкой)")
ax.axhline(0, color="k", lw=.7); ax.axvline(92, color="grey", lw=.6)
ax.legend(); ax.set_title("Сценарий C: конверсия пробитой стороны в бабочку ограничивает убыток")

Конверсия срезает максимальный убыток с ~\$436 до ~\$276 **и** строит палатку прибыли
примерно на \$221, если DEMO стабилизируется около 92.5. POP падает (бабочка выигрывает только в
узкой полосе) — это честный размен: вы платите дебет, чтобы *ограничить катастрофу* и купить зону
восстановления, а не чтобы повысить свои шансы.

## Дельта-хеджирование и решение на 21 DTE

Протестированный короткий пут нёс **+63 дельты** — уходя в ITM, он дрейфует в лонг. Если ваш взгляд
по-прежнему диапазонный, а не направленный, **захеджируйте дельту**, зашортив акции (по 1 дельте
каждая), чтобы вернуть книгу к нулю, вместо того чтобы роллировать.

In [ ]:
d = greeks.position_greeks(orig_now, 92, 0.32).delta
hedged = strategies.custom(OptionLeg("put",95,25/365,-1,1.58), StockLeg(-round(d), 92.0),
                           label="короткий пут + хедж акцией")
print("дельта без хеджа $:", r(d), "-> дельта с хеджем $:", r(greeks.position_greeks(hedged, 92, 0.32).delta))

И **правило 21 DTE**: позиции с короткой премией несут отрицательную гамму, которая в
последние недели звереет. Примерно на 21 DTE вы *принимаете решение* — забрать победителя,
роллировать всё целиком в новый цикл за кредит или закрыть сломанного проигравшего. Короткую гамму в
последнюю неделю не тащат в надежде на лучшее.

## Эксперименты

1. Сценарий A: попробуйте ролл в **92.5**-й пут на 120 DTE вместо 90-го. Кредита больше, но
   достаточно ли это понижает ваш страйк обязательства? Перепроверьте `roll_net`, POP и дельту.
2. Сценарий A: сделайте движение хуже — задайте `spot=88`, `vol=0.36`. Существует ли ещё ролл за
   кредит, или это уже ситуация *закрытия* (тезис сломан)?
3. Сценарий B: продолжайте роллировать нетронутую колл-сторону до 100/105. Смотрите, как рушится
   верхняя точка безубыточности: в какой момент «безопасная» сторона становится новой
   протестированной?
4. Сценарий C: сделайте конверсию **раньше**, при споте 92.5 и 25 DTE. Дебет меньше? Лучше ли стоит
   палатка бабочки относительно цены?
5. Захеджируйте дельту **путом** вместо акции: купите дешёвый OTM-пут, чтобы добавить отрицательной
   дельты. Сравните эффект на гамму и тету с хеджем акцией (`position_greeks` с греками `which`).